# Evaluating Scorio Math with `scorio.eval`

This notebook evaluates `gpt-oss-20b_medium` on a 10-question AIME 2026 window. Scorio
expects an `M x N` matrix: questions by sampled attempts. Here that is 10 by 80.

Only identity and correctness columns are read from the public Bucket. The top-20 token
distributions are not transferred. Comparing all four configurations opens 40 remote
Parquet files, so runtime depends on network latency.


In [1]:
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

from scorio import eval

BUCKET_ROOT = "hf://buckets/harimo/scorio-math"


def pool_path(model, task, question_id):
    return f"{BUCKET_ROOT}/data/{model}/{task}/q{question_id:02d}.parquet"


def read_pools(model, task, question_ids, columns, max_workers=2):
    """Read selected columns from question files, preserving question order."""
    paths = [pool_path(model, task, q) for q in question_ids]

    def read_one(path):
        return pq.read_table(path, columns=columns)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        tables = list(executor.map(read_one, paths))
    return pa.concat_tables(tables).to_pandas()

models = ["Qwen3.6-35B-A3B", "gpt-oss-20b_low", "gpt-oss-20b_medium", "gpt-oss-20b_high"]
model_name = "gpt-oss-20b_medium"
task = "aime_2026"
question_count = 10
question_ids = range(question_count)
columns = ["data_id", "seed", "evalscope_is_correct"]

rows = read_pools(model_name, task, question_ids, columns).sort_values(["data_id", "seed"])
assert rows.groupby("data_id").size().eq(80).all()
R = rows.evalscope_is_correct.to_numpy().astype(int).reshape(question_count, 80)

print(R.shape, R.dtype)
print(R[:3, :12])


(10, 80) int64
[[1 1 1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 0 1 1]]


## Bayes@N

Bayes@N estimates expected accuracy from `N` repeated attempts per question. It places a
uniform Beta(1, 1) prior on each question's success rate, which keeps finite-sample
estimates away from exact zero and one. `bayes_ci` returns the dataset-level posterior
mean, standard deviation, and a normal-approximation 95% credible interval.


In [2]:
mu, sigma, lo, hi = eval.bayes_ci(R)
print(f"Bayes@N: {mu:.3f} +- {sigma:.3f}")
print(f"95% credible interval: [{lo:.3f}, {hi:.3f}]")


Bayes@N: 0.891 +- 0.009
95% credible interval: [0.873, 0.910]


## Pass@k, Maj@k, and Pass^k

These metrics ask different questions: at least one success, strict-majority success, and
all-k success.


In [3]:
ks = [1, 2, 4, 8, 16, 80]
table = pd.DataFrame({
    "pass@k": [eval.pass_at_k(R, k) for k in ks],
    "maj@k": [eval.maj_at_k(R, k) for k in ks],
    "pass^k": [eval.pass_hat_k(R, k) for k in ks],
    "auc@k": [eval.auc_at_k(R, k) for k in ks],
}, index=pd.Index(ks, name="k"))

display(table.round(3))


,pass@k,maj@k,pass^k,auc@k
k,,,,
1,0.901,0.901,0.901,0.901
2,0.967,0.836,0.836,0.934
4,0.995,0.890,0.757,0.967
8,1.000,0.916,0.674,0.985
16,1.000,0.941,0.566,0.993
80,1.000,1.000,0.100,0.999


## Sample-budget sweep

Rows are ordered by seed, so slicing the first `n` columns gives a reproducible sample
budget.


In [4]:
budgets = [1, 2, 4, 8, 16, 32, 80]
sweep = pd.DataFrame(
    [eval.bayes_ci(R[:, :n]) for n in budgets],
    columns=["mu", "sigma", "lo", "hi"],
    index=pd.Index(budgets, name="samples"),
)
sweep["width"] = sweep.hi - sweep.lo
display(sweep.round(3))


,mu,sigma,lo,hi,width
samples,,,,,
1,0.667,0.075,0.521,0.813,0.292
2,0.725,0.062,0.603,0.847,0.244
4,0.783,0.048,0.690,0.877,0.186
8,0.850,0.033,0.785,0.915,0.130
16,0.878,0.022,0.834,0.922,0.088
32,0.900,0.015,0.871,0.929,0.059
80,0.891,0.009,0.873,0.910,0.037


## Compare all four model configurations


In [ ]:
matrices = []
for model in models:
    if model == model_name:
        matrices.append(R)
        continue
    one = read_pools(model, task, question_ids, columns).sort_values(["data_id", "seed"])
    matrices.append(one.evalscope_is_correct.to_numpy().astype(int).reshape(question_count, 80))

comparison = pd.DataFrame(
    [eval.bayes_ci(matrix) for matrix in matrices],
    columns=["mu", "sigma", "lo", "hi"],
    index=models,
)
display(comparison.sort_values("mu", ascending=False).round(3))


The [evaluation reference](https://github.com/mohsenhariri/scorio/blob/main/scorio/eval/README.md)
lists the remaining estimators and their sources.
